Este notebook tem como objetivo inspecionar a tabela e dicionario, criar as features necessárias e a estatística descritiva.

In [0]:
%sql

SELECT * FROM sandbox.prcg.transacoes LIMIT 5

datahora_entregue,duracao,distancia,vlr_pago,id_motorista,latitude_origem,longitude_origem,latitude_destino,longitude_destino
03/11/2025 07:12,12,4.2,23.97,1,-23.513742609587695,-46.62327810242303,-23.483853341186528,-46.648585835169115
23/10/2025 09:43,11,4.3,24.71,2,-23.51207921873963,-46.62739562527016,-23.474032512968865,-46.63578672625711
22/10/2025 11:12,9,4.2,23.01,1,-23.574913898123835,-46.60181176389887,-23.611927332177384,-46.592856334210495
18/10/2025 18:03,11,3.4,21.2,3,-23.581033986174024,-46.63696302378459,-23.598154937550955,-46.66461159876348
13/10/2025 11:54,11,3.6,21.74,3,-23.581544813632622,-46.643830193209304,-23.56791182574625,-46.611816490410796


In [0]:
%sql
WITH tbl_features AS (
  SELECT 
    day(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as dia,
    dayofweek(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as dia_semana,
    month(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as mes,
    hour(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as hora,
    duracao,
    distancia,
    vlr_pago,
    latitude_origem,
    longitude_origem,
    latitude_destino,
    longitude_destino
  FROM sandbox.prcg.transacoes
)
SELECT 
  dia,
  dia_semana,
  mes,
  hora,
  case when hora < 12 and hora > 6 then 1
       when hora >= 12 and hora <= 18 then 2
       else 3 end as periodo,
  duracao,
  distancia,
  vlr_pago,
  latitude_origem,
  longitude_origem,
  latitude_destino,
  longitude_destino
FROM tbl_features

dia,dia_semana,mes,hora,periodo,duracao,distancia,vlr_pago,latitude_origem,longitude_origem,latitude_destino,longitude_destino
3,2,11,7,1,12,4.2,23.97,-23.513742609587695,-46.62327810242303,-23.483853341186528,-46.648585835169115
23,5,10,9,1,11,4.3,24.71,-23.51207921873963,-46.62739562527016,-23.474032512968865,-46.63578672625711
22,4,10,11,1,9,4.2,23.01,-23.574913898123835,-46.60181176389887,-23.611927332177384,-46.592856334210495
18,7,10,18,2,11,3.4,21.2,-23.581033986174024,-46.63696302378459,-23.598154937550955,-46.66461159876348
13,2,10,11,1,11,3.6,21.74,-23.581544813632622,-46.643830193209304,-23.56791182574625,-46.611816490410796
11,7,10,11,1,8,3.8,23.26,-23.52960821981363,-46.66281396280993,-23.499799594780598,-46.64438846657277
11,7,10,8,1,9,4.2,24.74,-23.585069168120537,-46.61634743301252,-23.621745269018117,-46.62681068452422
10,6,10,19,3,106,4.9,27.0,-23.542660608006138,-46.680536511354774,-23.499054094285043,-46.68864739163299
10,6,10,18,2,15,5.1,27.82,-23.595103808176866,-46.64581113130338,-23.587440347311027,-46.59654247057943
7,3,10,11,1,9,3.4,20.71,-23.56826141657061,-46.660487972079046,-23.58510671249467,-46.68833247475998


In [0]:
!pip install haversine

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from haversine import haversine
from pyspark.sql.types import FloatType

def dist_haversine(lat_o, lon_o, lat_d, lon_d):
    return haversine((lat_o, lon_o), (lat_d, lon_d))

spark.udf.register("dist_haversine", dist_haversine, FloatType())

<function __main__.dist_haversine(lat_o, lon_o, lat_d, lon_d)>

In [0]:
%sql

SELECT 
  dist_haversine(latitude_origem, longitude_origem, latitude_destino, longitude_destino) as distancia
FROM sandbox.prcg.transacoes LIMIT 5

distancia
4.2078524
4.3162794
4.215667
3.4003665
3.5976384


In [0]:

spark.sql("""
    WITH tbl_features AS (
    SELECT 
        day(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as dia,
        dayofweek(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as dia_semana,
        month(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as mes,
        hour(to_timestamp(datahora_entregue, 'dd/MM/yyy HH:mm')) as hora,
        duracao,
        vlr_pago,
        latitude_origem,
        longitude_origem,
        latitude_destino,
        longitude_destino
    FROM sandbox.prcg.transacoes
    )
    SELECT 
    dia,
    dia_semana,
    mes,
    hora,
    case when hora < 12 and hora > 6 then 1
        when hora >= 12 and hora <= 18 then 2
        else 3 end as periodo,
    duracao,
    vlr_pago,
    latitude_origem,
    longitude_origem,
    latitude_destino,
    longitude_destino,
    dist_haversine(latitude_origem, longitude_origem, latitude_destino, longitude_destino) as distancia
    FROM tbl_features
""").createOrReplaceTempView("tbl_features")

In [0]:
spark.sql(
    """
    SELECT 
        vlr_pago,
        dia,
        dia_semana,
        mes,
        hora,
        periodo,
        distancia,
        latitude_origem,
        longitude_origem,
        latitude_destino,
        longitude_destino
    FROM tbl_features
"""
).createOrReplaceTempView("tbl_ml")

In [0]:
spark.sql(
    """
    SELECT 
        vlr_pago,
        dia,
        dia_semana,
        mes,
        hora,
        periodo,
        distancia,
        duracao
    FROM tbl_features
"""
).createOrReplaceTempView("tbl_eda")

In [0]:
df_eda = spark.table("tbl_eda").toPandas()
df_ml = spark.table("tbl_ml").toPandas()

In [0]:
df_eda.describe().T

,count,mean,std,min,25%,50%,75%,max
vlr_pago,104.0,22.817692,3.371668,14.3400,20.792500,23.330000,24.84250,33.120000
dia,104.0,14.567308,9.321407,1.0000,7.000000,13.000000,23.00000,31.000000
dia_semana,104.0,4.269231,1.807378,1.0000,2.750000,4.000000,6.00000,7.000000
mes,104.0,8.182692,2.079789,2.0000,7.000000,8.000000,10.00000,11.000000
hora,104.0,12.990385,3.674882,7.0000,10.000000,13.000000,16.00000,22.000000
periodo,104.0,1.634615,0.639443,1.0000,1.000000,2.000000,2.00000,3.000000
distancia,104.0,3.849774,0.863909,1.6973,3.373871,4.054581,4.39638,6.293622
duracao,104.0,12.211538,10.319811,6.0000,8.750000,11.000000,13.00000,106.000000


In [0]:
df_ml.describe().T

,count,mean,std,min,25%,50%,75%,max
vlr_pago,104.0,22.817692,3.371668,14.340000,20.792500,23.330000,24.842500,33.120000
dia,104.0,14.567308,9.321407,1.000000,7.000000,13.000000,23.000000,31.000000
dia_semana,104.0,4.269231,1.807378,1.000000,2.750000,4.000000,6.000000,7.000000
mes,104.0,8.182692,2.079789,2.000000,7.000000,8.000000,10.000000,11.000000
hora,104.0,12.990385,3.674882,7.000000,10.000000,13.000000,16.000000,22.000000
periodo,104.0,1.634615,0.639443,1.000000,1.000000,2.000000,2.000000,3.000000
distancia,104.0,3.849774,0.863909,1.697300,3.373871,4.054581,4.396380,6.293622
latitude_origem,104.0,-23.552737,0.026101,-23.604522,-23.575985,-23.552217,-23.529123,-23.503112
longitude_origem,104.0,-46.632568,0.026360,-46.680537,-46.657301,-46.634099,-46.608650,-46.587278
latitude_destino,104.0,-23.552667,0.040540,-23.634857,-23.583268,-23.550637,-23.519311,-23.472734


In [0]:
df_eda[df_eda["duracao"] > 100]

,vlr_pago,dia,dia_semana,mes,hora,periodo,distancia,duracao
7,27.0,10,6,10,19,3,4.918842,106
